# Coffee Sales Analysis - Exploratory Data Analysis (EDA) & Testing

> **Dành cho thành viên dự án:** Sử dụng notebook này để khám phá dữ liệu, kiểm tra lỗi, và thử nghiệm các biểu đồ trước khi đóng gói code vào `src/`.

### Mục tiêu chính:
1. Khám phá cấu trúc dữ liệu thô trong `data/raw/`.
2. Kiểm tra missing values, dữ liệu ngoại lai, định dạng cột.
3. Thử nghiệm logic làm sạch dữ liệu.
4. Test các dạng biểu đồ trả lời 2 câu hỏi nghiệp vụ:
   - *Thời điểm nào bán chạy?*
   - *Sản phẩm nào mang lại doanh thu cao?*

In [1]:
# 1. Khai báo các thư viện cần thiết
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# Đường dẫn thư mục dữ liệu
RAW_DIR = Path("../data/raw")
CLEAN_DIR = Path("../data/clean")

## 2. Đọc & Khảo sát Dữ liệu Thô (data/raw/)

In [2]:
# Đọc dữ liệu thô và kiểm tra cấu trúc tổng quan.
raw_path = RAW_DIR / "coffee_shop_sales.csv"
df_raw = pd.read_csv(raw_path)

print(f"Kích thước dữ liệu: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")
print("\nTên cột:")
print(df_raw.columns.tolist())
print("\nNăm dòng đầu:")
display(df_raw.head())
print("\nThông tin kiểu dữ liệu:")
df_raw.info()
print("\nThống kê số:")
display(df_raw.describe(include="all").transpose())

Kích thước dữ liệu: 149,116 dòng x 11 cột

Tên cột:
['transaction_id', 'transaction_date', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail']

Năm dòng đầu:


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,01-01-2023,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,01-01-2023,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,01-01-2023,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,01-01-2023,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,01-01-2023,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg



Thông tin kiểu dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    149116 non-null  int64  
 1   transaction_date  149116 non-null  str    
 2   transaction_time  149116 non-null  str    
 3   transaction_qty   149116 non-null  int64  
 4   store_id          149116 non-null  int64  
 5   store_location    149116 non-null  str    
 6   product_id        149116 non-null  int64  
 7   unit_price        149116 non-null  float64
 8   product_category  149116 non-null  str    
 9   product_type      149116 non-null  str    
 10  product_detail    149116 non-null  str    
dtypes: float64(1), int64(4), str(6)
memory usage: 12.5 MB

Thống kê số:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
transaction_id,149116.0,NaN,NaN,NaN,74737.371872,43153.600016,1.0,37335.75,74727.5,112094.25,149456.0
transaction_date,149116,181,19-06-2023,1343,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transaction_time,149116,25762,09:31:15,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN
transaction_qty,149116.0,NaN,NaN,NaN,1.438276,0.542509,1.0,1.0,1.0,2.0,8.0
store_id,149116.0,NaN,NaN,NaN,5.342063,2.074241,3.0,3.0,5.0,8.0,8.0
store_location,149116,3,Hell's Kitchen,50735,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,149116.0,NaN,NaN,NaN,47.918607,17.93002,1.0,33.0,47.0,60.0,87.0
unit_price,149116.0,NaN,NaN,NaN,3.382219,2.658723,0.8,2.5,3.0,3.75,45.0
product_category,149116,9,Coffee,58416,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_type,149116,29,Brewed Chai tea,17183,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Kiểm tra Missing Values & Bất thường (Outliers)

In [5]:
# Kiểm tra missing, duplicate và các giá trị ngoài quy chuẩn trong data dictionary.
expected_columns = {
    "transaction_id", "transaction_date", "transaction_time",
    "transaction_qty", "store_id", "store_location", "product_id",
    "unit_price", "product_category", "product_type", "product_detail",
}

print("Các cột thiếu so với data dictionary:", sorted(expected_columns - set(df_raw.columns)))
print("Các cột chưa được mô tả:", sorted(set(df_raw.columns) - expected_columns))
print("\nMissing values theo cột:")
display(df_raw.isna().sum().to_frame("missing_count"))
print(f"Số dòng trùng hoàn toàn: {df_raw.duplicated().sum():,}")
print(f"Số transaction_id trùng: "
      f"{df_raw['transaction_id'].duplicated().sum():,}")

numeric_check = df_raw[["transaction_qty", "unit_price"]].apply(pd.to_numeric, errors="coerce")
print("\nKiểm tra miền giá trị:")
print(
    f"transaction_qty: "
    f"{numeric_check['transaction_qty'].min()} -> "
    f"{numeric_check['transaction_qty'].max()}"
)
print(
    f"Số quantity <= 0: "
    f"{(numeric_check['transaction_qty'] <= 0).sum():,}"
)
print(
    f"Số unit_price < 0: "
    f"{(numeric_check['unit_price'] < 0).sum():,}"
)

parsed_dates = pd.to_datetime(df_raw["transaction_date"], dayfirst=True, errors="coerce")
parsed_times = pd.to_datetime(df_raw["transaction_time"], format="mixed", errors="coerce")
print(f"\nKhoảng ngày: {parsed_dates.min().date()} -> {parsed_dates.max().date()}")
print(f"Khoảng giờ: {parsed_times.min().time()} -> {parsed_times.max().time()}")
print("Chi nhánh:", sorted(df_raw["store_location"].dropna().unique()))
print("Danh mục:", sorted(df_raw["product_category"].dropna().unique()))

Các cột thiếu so với data dictionary: []
Các cột chưa được mô tả: []

Missing values theo cột:


,missing_count
transaction_id,0
transaction_date,0
transaction_time,0
transaction_qty,0
store_id,0
store_location,0
product_id,0
unit_price,0
product_category,0
product_type,0


Số dòng trùng hoàn toàn: 0
Số transaction_id trùng: 0

Kiểm tra miền giá trị:
transaction_qty: 1 -> 8
Số quantity <= 0: 0
Số unit_price < 0: 0

Khoảng ngày: 2023-01-01 -> 2023-06-30
Khoảng giờ: 06:00:00 -> 20:59:32
Chi nhánh: ['Astoria', "Hell's Kitchen", 'Lower Manhattan']
Danh mục: ['Bakery', 'Branded', 'Coffee', 'Coffee beans', 'Drinking Chocolate', 'Flavours', 'Loose Tea', 'Packaged Chocolate', 'Tea']


## 4. Thử nghiệm Làm sạch & Tạo đặc trưng mới (Feature Engineering)

In [ ]:
# TODO [Thành viên]: Chuyển đổi datetime, tách giờ (hour), tách thứ (day_name), tính doanh thu (revenue)

## 5. Thử nghiệm Biểu đồ: 'Thời điểm nào bán chạy?'

In [ ]:
# TODO [Thành viên]: Thử nghiệm vẽ biểu đồ theo giờ, theo ngày trong tuần, hoặc heatmap

## 6. Thử nghiệm Biểu đồ: 'Sản phẩm nào mang lại doanh thu cao?'

In [ ]:
# TODO [Thành viên]: Thử nghiệm vẽ Top sản phẩm, tỷ trọng danh mục, tương quan giá vs sản lượng